# Milestone 1 — Debug a Silently Failing Training Run
## Problem:
    A binary classifier on tabular data (10,000 rows) shows steadily falling training loss, but validation accuracy stays near 55% and fluctuates between epochs.

Provided training code:

In [ ]:
X = StandardScaler().fit_transform(X)
X_tr, X_val = X[:8000], X[8000:]
y_tr, y_val = y[:8000], y[8000:]
np.random.shuffle(X_tr)

model = nn.Sequential(nn.Linear(40, 128), nn.ReLU(), nn.Dropout(0.5),
              nn.BatchNorm1d(128), nn.Linear(128, 2))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=1, gamma=0.9)

for epoch in range(50):
    model.train()
    for xb, yb in train_loader:
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        scheduler.step()
    val_logits = model(torch.tensor(X_val))
    print(epoch, accuracy(val_logits, y_val))

In [ ]:
# Bug No:1 
# Missing Imports 
#fix import all necessory libraries

import numpy as np
import torch
import pandas as pd
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

# Bug No:2 Missing Data load
# fix : load dataset accordingly

path = "data path"
X = pd.read_csv(path)
y = X['label']


# Bug No:3  standardize the data and the spliting
# Fix :First split then Standardize  and use train test split from sklearn

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0)


# Bug No:4  applying fit_transfrom to both train and val dataset 
# fix : fit the just training data and transfrom both train and val ++++++ fit_transform train and transform val

scaler = StandardScaler().fit(X_tr)
X_tr, X_val = scaler.transform(X_tr), scaler.transform(X_val)


#Bug No: 5   suffling just X(features ) not labels 
# This changes the order of the feature rows but leaves their labels unchanged.
# The features and labels no longer correspond to the same examples.
# fix drop the manual shuffle and use DataLoader(shuffle=True)

# np.random.shuffle(X_tr)


# Bug No:6 Missing Dataloader object 
# Fix : convert numpy array into tensor and create Dataloader object


X_tr_tensor = torch.tensor(X_tr).float()
X_val_tensor = torch.tensor(X_val).float()

y_tr_tensor = torch.tensor(y_tr.values).long()
y_val_tensor = torch.tensor(y_val.values).long()

train_dataset = TensorDataset(X_tr_tensor, y_tr_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)


BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


# Bug 11 batch norm after RELU and Dropout 
# fix use Batch Norm before Relu and Dropout 

# model = nn.Sequential(
#     nn.Linear(40, 128),
#     nn.ReLU(),
#     nn.Dropout(0.5),
#     nn.BatchNorm1d(128), 
#     nn.Linear(128, 2))

model = nn.Sequential(
    nn.Linear(40, 128),
    nn.BatchNorm1d(128), 
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 2))


# Bug No:7  Missing Loss_Function
#  Fix : Use a loss function according to task 


loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)


# Bug No: 8 torch.optim.lr_scheduler.StepLR   reduces the learning rate after every and Multiply the current learning rate by 0.9
# after 50 epoch lr will be very small 

# scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=1, gamma=0.9)
# Fix use or just dont use a scheduler 

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)





for epoch in range(50):
    # Training
    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        opt.zero_grad()

        logits = model(xb)
        loss = loss_fn(logits, yb.long())

        loss.backward()
        opt.step()

        train_loss += loss.item() * xb.size(0)

        preds = logits.argmax(dim=1)
        train_correct += (preds == yb).sum().item()
        train_total += yb.size(0)

    train_loss /= len(train_loader.dataset)
    train_acc =  train_correct / train_total


# Bug No:9  No model.eval() / torch.no_grad() at validation

    # fix Use proper  Validation
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for xval,yval in val_loader:
            val_logits = model(xval)
            loss = loss_fn(val_logits, y_val_tensor)
            val_loss = loss.item()

            val_acc = (
                (val_logits.argmax(dim=1) == y_val_tensor)
                .float()
                .mean()
                .item()
            )

    # Bug N0:10 scheduler.step is used after every batch   
    # fix use  scheduler.step use Once per epoch         
    scheduler.step()

    print(
        f"Epoch {epoch + 1:02d}/50 | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_acc:.4f} | "
        f"LR: {opt.param_groups[0]['lr']:.6f}"
    )

# Corrected Code 

In [ ]:
import numpy as np
import torch
import pandas as pd
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

path = "data path"
X = pd.read_csv(path)
y = X['label']


X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0)

scaler = StandardScaler().fit(X_tr)
X_tr, X_val = scaler.transform(X_tr), scaler.transform(X_val)

X_tr_tensor = torch.tensor(X_tr).float()
X_val_tensor = torch.tensor(X_val).float()

y_tr_tensor = torch.tensor(y_tr.values).long()
y_val_tensor = torch.tensor(y_val.values).long()

train_dataset = TensorDataset(X_tr_tensor, y_tr_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)


BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


model = nn.Sequential(
    nn.Linear(40, 128),
    nn.BatchNorm1d(128), 
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 2))


loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)


for epoch in range(50):
    # Training
    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        opt.zero_grad()

        logits = model(xb)
        loss = loss_fn(logits, yb.long())

        loss.backward()
        opt.step()

        train_loss += loss.item() * xb.size(0)

        preds = logits.argmax(dim=1)
        train_correct += (preds == yb).sum().item()
        train_total += yb.size(0)

    train_loss /= len(train_loader.dataset)
    train_acc =  train_correct / train_total


    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for xval,yval in val_loader:
            val_logits = model(xval)
            loss = loss_fn(val_logits, y_val_tensor)
            val_loss = loss.item()

            val_acc = (
                (val_logits.argmax(dim=1) == y_val_tensor)
                .float()
                .mean()
                .item()
            )
        
    scheduler.step()

    print(
        f"Epoch {epoch + 1:02d}/50 | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_acc:.4f} | "
        f"LR: {opt.param_groups[0]['lr']:.6f}"
    )

# 4 Describe quick experiments that would reveal pipeline problems without access to the code.


# 1. Establish a Majority-Class Baseline

I will check the class distribution in the training and validation datasets and calculate the accuracy of always predicting the most frequent class. If the model's accuracy is close to this baseline, it may not be learning useful patterns.

# 2. Compare Training and Validation Performance

I will monitor both training and validation loss and accuracy. If training performance improves while validation performance remains poor, the model may be overfitting, or there may be a problem with the data split or preprocessing.

# 3. Test on a Small Subset

I will train the model on a small subset of around 32–64 samples. The model should be able to achieve very high training accuracy if the training pipeline is working correctly. If it cannot, there may be problems with the labels, data preprocessing, or training process.

# 4. Compare Learning Rates

I will train the model with a fixed learning rate and compare the results with the current scheduler. If the fixed learning rate gives better results, the scheduler may be reducing the learning rate too quickly.